# Phase 1: Web Scraping – Book Market Data Collection

## Project Title
**Book Market Analysis and Rating Prediction Using Web Scraping and Machine Learning**

## Objective
The objective of this phase is to collect book-related information from the public Books to Scrape website using Python web-scraping techniques.

The collected data will be used for data cleaning, exploratory data analysis, machine learning, and interactive dashboard development.

In [1]:
%pip install requests beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Importing Required Libraries

The project uses Python libraries for sending HTTP requests, parsing HTML pages, managing tabular data, and controlling the scraping process.

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

## 2. Accessing the Website

The Books to Scrape website is accessed using the Requests library.

The HTTP response status code is checked to confirm that the website is successfully accessible before starting the scraping process.

In [5]:
url = "https://books.toscrape.com/"
response = requests.get(url)

print("Status Code:", response.status_code)

Status Code: 200


## 3. Parsing the Web Page

BeautifulSoup is used to parse the HTML content of the webpage.

This allows the required book information to be located and extracted from the page structure.

In [6]:
soup = BeautifulSoup(response.text, "html.parser")

print("Page title:", soup.title.text.strip())

Page title: All products | Books to Scrape - Sandbox


## 4. Identifying Book Records

Each book is represented by a product container on the catalogue page.

The scraper identifies these product containers and determines the number of books available on the current page.

In [7]:
books = soup.find_all("article", class_="product_pod")

print("Books found:", len(books))


Books found: 20


## 5. Extracting Basic Book Information

The following fields are extracted from each catalogue listing:

- Book title
- Price
- Rating
- Availability

In [8]:
book = books[0]

title = book.h3.a["title"]
price = book.find("p", class_="price_color").text.strip()
rating = book.find("p", class_="star-rating")["class"][1]
availability = book.find("p", class_="instock").text.strip()

print("Title:", title)
print("Price:", price)
print("Rating:", rating)
print("Availability:", availability)

Title: A Light in the Attic
Price: Â£51.77
Rating: Three
Availability: In stock


## 6. Constructing Product URLs

The individual product-page links are converted into complete URLs using `urljoin()`.

These URLs are later used to access the detailed information for each book.

In [11]:
from urllib.parse import urljoin

product_url = urljoin(url, book.h3.a["href"])

print("Product URL:", product_url)

Product URL: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


## 7. Accessing Individual Book Pages

Each book has a separate product-detail page.

The scraper sends a request to the product URL to retrieve additional book information that is not available on the catalogue page.

In [12]:
detail_response = requests.get(product_url)

print("Status Code:", detail_response.status_code)

Status Code: 200


## 8. Parsing Detailed Book Information

The individual product page is parsed using BeautifulSoup so that additional attributes can be extracted from the page.

In [13]:
detail_soup = BeautifulSoup(detail_response.text, "html.parser")

print(detail_soup.title.text.strip())

A Light in the Attic | Books to Scrape - Sandbox


## 9. Extracting Book Category

The book category is extracted from the breadcrumb navigation on the individual product page.

In [14]:
category = detail_soup.select("ul.breadcrumb li a")[-1].text.strip()

print("Category:", category)

Category: Poetry


## 10. Extracting UPC

The Universal Product Code (UPC) is extracted from the product information table.

The UPC provides a unique identifier for each book.

In [15]:
upc = detail_soup.find("th", string="UPC").find_next_sibling("td").text.strip()

print("UPC:", upc)

UPC: a897fe39b1053632


## 11. Extracting Availability and Stock Information

The detailed availability information is extracted from the product page.

This information is used to determine the number of books currently available in stock.

In [16]:
availability_detail = detail_soup.select_one("p.availability").text.strip()

print("Availability:", availability_detail)

Availability: In stock (22 available)


## 12. Extracting Stock Count

Regular expressions are used to extract the numeric stock count from the availability text.

The extracted value is converted into an integer for later analysis.

In [17]:
import re

stock_match = re.search(r"\((\d+) available\)", availability_detail)

if stock_match:
    stock_count = int(stock_match.group(1))
else:
    stock_count = 0

print("Stock Count:", stock_count)

Stock Count: 22


## 13. Extracting Tax Information

The tax value is extracted from the product information table on the individual book page.

In [18]:
tax = detail_soup.find("th", string="Tax").find_next_sibling("td").text.strip()

print("Tax:", tax)

Tax: Â£0.00


## 14. Extracting Book Description

The book description is extracted from the product-detail page.

If a description is not available, an empty value is recorded so that the scraping process can continue without interruption.

In [19]:
description = detail_soup.select_one("#product_description")

if description:
    description = description.find_next_sibling("p").text.strip()
else:
    description = ""

print("Description:", description)

Description: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your cradle, too?Baby, I think someone down here'sGot it in for you. Shel, you 

## 15. Combining Book Attributes

All extracted attributes are combined into a single dictionary representing one book record.

The collected attributes include title, price, rating, availability, product URL, category, UPC, stock count, tax, and description.

In [20]:
book_data = {
    "title": title,
    "price": price,
    "rating": rating,
    "availability": availability,
    "product_url": product_url,
    "category": category,
    "upc": upc,
    "stock_count": stock_count,
    "tax": tax,
    "description": description
}

for key, value in book_data.items():
    print(f"{key}: {value}")

title: A Light in the Attic
price: Â£51.77
rating: Three
availability: In stock
product_url: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
category: Poetry
upc: a897fe39b1053632
stock_count: 22
tax: Â£0.00
description: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that

## 16. Scraping All Catalogue Pages

The scraper iterates through the catalogue pages using the website's pagination links.

For every page, it:

1. Sends an HTTP request.
2. Parses the HTML content.
3. Identifies all book listings.
4. Extracts basic book information.
5. Constructs the product URL.
6. Moves to the next catalogue page.

A short delay is included between requests to reduce the request rate.

The scraping process successfully collected **1,000 books from 50 catalogue pages**.

In [21]:
all_books = []

page_url = "https://books.toscrape.com/catalogue/page-1.html"

while page_url:
    print("Scraping:", page_url)

    page_response = requests.get(page_url)
    page_soup = BeautifulSoup(page_response.text, "html.parser")

    books = page_soup.find_all("article", class_="product_pod")

    for book in books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text.strip()
        rating = book.find("p", class_="star-rating")["class"][1]
        availability = book.find("p", class_="instock").text.strip()
        product_url = urljoin(page_url, book.h3.a["href"])

        all_books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "product_url": product_url
        })

    next_button = page_soup.select_one("li.next a")

    if next_button:
        page_url = urljoin(page_url, next_button["href"])
    else:
        page_url = None

    time.sleep(0.5)

print("Total books collected:", len(all_books))

Scraping: https://books.toscrape.com/catalogue/page-1.html
Scraping: https://books.toscrape.com/catalogue/page-2.html
Scraping: https://books.toscrape.com/catalogue/page-3.html
Scraping: https://books.toscrape.com/catalogue/page-4.html
Scraping: https://books.toscrape.com/catalogue/page-5.html
Scraping: https://books.toscrape.com/catalogue/page-6.html
Scraping: https://books.toscrape.com/catalogue/page-7.html
Scraping: https://books.toscrape.com/catalogue/page-8.html
Scraping: https://books.toscrape.com/catalogue/page-9.html
Scraping: https://books.toscrape.com/catalogue/page-10.html
Scraping: https://books.toscrape.com/catalogue/page-11.html
Scraping: https://books.toscrape.com/catalogue/page-12.html
Scraping: https://books.toscrape.com/catalogue/page-13.html
Scraping: https://books.toscrape.com/catalogue/page-14.html
Scraping: https://books.toscrape.com/catalogue/page-15.html
Scraping: https://books.toscrape.com/catalogue/page-16.html
Scraping: https://books.toscrape.com/catalogue/pa

## 17. Scraping Detailed Book Information

After collecting the catalogue records, the scraper visits each individual product page.

The following additional fields are extracted:

- Category
- UPC
- Detailed availability
- Stock count
- Tax
- Description

Error handling is included so that an issue with one product page does not stop the complete scraping process.

In [22]:
for i, book in enumerate(all_books):
    try:
        detail_response = requests.get(book["product_url"], timeout=10)
        detail_soup = BeautifulSoup(detail_response.text, "html.parser")

        # Category
        book["category"] = detail_soup.select("ul.breadcrumb li a")[-1].text.strip()

        # UPC
        book["upc"] = detail_soup.find(
            "th", string="UPC"
        ).find_next_sibling("td").text.strip()

        # Availability and stock count
        availability_detail = detail_soup.select_one(
            "p.availability"
        ).text.strip()

        book["availability"] = availability_detail

        stock_match = re.search(
            r"\((\d+) available\)", availability_detail
        )

        book["stock_count"] = int(stock_match.group(1)) if stock_match else 0

        # Tax
        book["tax"] = detail_soup.find(
            "th", string="Tax"
        ).find_next_sibling("td").text.strip()

        # Description
        description_tag = detail_soup.select_one("#product_description")

        if description_tag:
            book["description"] = description_tag.find_next_sibling("p").text.strip()
        else:
            book["description"] = ""

    except Exception as e:
        print(f"Error at book {i + 1}: {e}")

    if (i + 1) % 100 == 0:
        print(f"{i + 1} books processed")

    time.sleep(0.2)

print("Detailed scraping completed!")

100 books processed
200 books processed
300 books processed
400 books processed
500 books processed
600 books processed
700 books processed
800 books processed
900 books processed
1000 books processed
Detailed scraping completed!


## 18. Creating the Scraped Dataset

The collected book records are converted into a Pandas DataFrame.

The resulting dataset contains **1,000 rows and 10 columns**.

In [23]:
df = pd.DataFrame(all_books)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()

Rows: 1000
Columns: 10


,title,price,rating,availability,product_url,category,upc,stock_count,tax,description
0,A Light in the Attic,Â£51.77,Three,In stock (22 available),https://books.toscrape.com/catalogue/a-light-i...,Poetry,a897fe39b1053632,22,Â£0.00,It's hard to imagine a world without A Light i...
1,Tipping the Velvet,Â£53.74,One,In stock (20 available),https://books.toscrape.com/catalogue/tipping-t...,Historical Fiction,90fa61229261140a,20,Â£0.00,"""Erotic and absorbing...Written with starling ..."
2,Soumission,Â£50.10,One,In stock (20 available),https://books.toscrape.com/catalogue/soumissio...,Fiction,6957f44c3847a760,20,Â£0.00,"Dans une France assez proche de la nÃ´tre, un ..."
3,Sharp Objects,Â£47.82,Four,In stock (20 available),https://books.toscrape.com/catalogue/sharp-obj...,Mystery,e00eb4fd7b871a48,20,Â£0.00,"WICKED above her hipbone, GIRL across her hear..."
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock (20 available),https://books.toscrape.com/catalogue/sapiens-a...,History,4165285e1663650f,20,Â£0.00,From a renowned historian comes a groundbreaki...


## 19. Initial Data Validation

The dataset is inspected to verify its column structure and identify missing values.

This initial validation helps confirm that the scraping process produced a complete dataset before saving the raw data.

In [24]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

Column names:
['title', 'price', 'rating', 'availability', 'product_url', 'category', 'upc', 'stock_count', 'tax', 'description']

Missing values:
title           0
price           0
rating          0
availability    0
product_url     0
category        0
upc             0
stock_count     0
tax             0
description     0
dtype: int64


## 20. Saving the Raw Dataset

The scraped dataset is saved as:

`raw_books_data.csv`

The raw dataset is preserved before any cleaning or transformation is performed.

In [25]:
raw_file = "raw_books_data.csv"

df.to_csv(raw_file, index=False, encoding="utf-8-sig")

print(f"Raw dataset saved successfully as: {raw_file}")
print(f"Total records: {len(df)}")

Raw dataset saved successfully as: raw_books_data.csv
Total records: 1000
